# CSPF — NLI-based fidelity (replaces the degenerate Φ)
Scores every sampled (original, perturbed) pair with an off-the-shelf NLI model in **both directions** (original⇒perturbed and perturbed⇒original). A pair is *semantically equivalent* when both directions are entailments: `nli_bi = min(P_entail(o→p), P_entail(p→o))`.
Then: (a) per-family fidelity profile, (b) flip rate restricted to NLI-equivalent pairs (does the instability survive when only genuinely equivalent pairs are kept?), (c) agreement of NLI with the human audit (AUC), compared with Φ (AUC ≈ 0.49).
- **No API.** Model is a public HF checkpoint (default `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`; fallback `microsoft/deberta-large-mnli`).
- **Runtime (T4):** default stratified sample ≈ 45k pairs → 90k NLI passes ≈ 10–15 min. `FULL_RUN=True` scores all 807,851 pairs (≈ 3–4 h; resumable, chunks cached on Drive).
- Needs `ALL_MODELS_combined_FIXED.csv` (from `CSPF_fix_alignment.ipynb`) and `annotated_output.csv` (your 576 audited pairs).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"
HUMAN_CSV = f"{BASE}/annotated_output.csv"
OUT_DIR   = f"{BASE}/results_FIXED/nli"
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
FULL_RUN  = False          # True = all pairs
PER_STRATUM = 1500         # sample size per (family x severity) stratum when FULL_RUN=False
CHUNK = 20000; BATCH = 128; MAXLEN = 160; SEED = 0
import os; os.makedirs(OUT_DIR+"/chunks", exist_ok=True)

In [ ]:
!pip -q install sentencepiece
import pandas as pd, numpy as np, os, glob, warnings; warnings.filterwarnings("ignore")
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist()
MODELS = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
print(len(MODELS), "models")
meta = ["source_dataset","review_idx","original_review","perturbed_review","perturbation_family","severity","sem_preservation_score"]
use = meta + [f"{m}_{s}_label" for m in MODELS for s in ("orig","pert")]
parts=[]
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    fl = np.zeros(len(ch), dtype=np.float32)
    for m in MODELS: fl += (ch[f"{m}_orig_label"].values != ch[f"{m}_pert_label"].values)
    ch = ch[meta].copy(); ch["mean_flip"] = fl/len(MODELS); parts.append(ch)
df = pd.concat(parts, ignore_index=True); del parts
df["key"] = df.source_dataset.astype(str)+"::"+df.review_idx.astype(str)
df["pair_id"] = np.arange(len(df))
df["stratum"] = df.perturbation_family+"|"+df.severity.astype(str)
N_str = df.groupby("stratum").size()
if FULL_RUN: S = df.copy()
else:
    sh = df.sample(frac=1, random_state=SEED); S = sh[sh.groupby("stratum").cumcount() < PER_STRATUM].copy()
n_s = S.groupby("stratum").size(); S["w"] = S.stratum.map(N_str/n_s)       # inverse-probability weight -> corpus-level estimates
S = S.reset_index(drop=True); print(len(df), "pairs in corpus;", len(S), "scored")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(NLI_MODEL); mdl = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(dev).eval()
if dev=="cuda": mdl.half()
l2i = {v.lower():k for k,v in mdl.config.id2label.items()}; E_, N_, C_ = l2i["entailment"], l2i["neutral"], l2i["contradiction"]
print("label map:", mdl.config.id2label)

@torch.no_grad()
def nli_probs(prem, hyp):
    """returns array [n,3] = P(entail), P(neutral), P(contradiction); length-sorted batches"""
    order = np.argsort([len(a)+len(b) for a,b in zip(prem,hyp)]); out = np.zeros((len(prem),3), dtype=np.float32)
    for i in range(0, len(order), BATCH):
        ix = order[i:i+BATCH]
        enc = tok([prem[j] for j in ix],[hyp[j] for j in ix], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(dev)
        p = torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy()
        out[ix] = p[:,[E_,N_,C_]]
    return out

def score(frame, tag):
    """resumable: one .npy per chunk on Drive"""
    res=[]
    for s in range(0, len(frame), CHUNK):
        f = f"{OUT_DIR}/chunks/{tag}_{s}.npy"
        if os.path.exists(f): res.append(np.load(f)); continue
        sub = frame.iloc[s:s+CHUNK]; o=sub.original_review.astype(str).tolist(); p=sub.perturbed_review.astype(str).tolist()
        a = nli_probs(o,p); b = nli_probs(p,o); arr = np.hstack([a,b]); np.save(f, arr); res.append(arr); print(tag, s, "done")
    return np.vstack(res)

In [ ]:
R = score(S, "main")
S[["ent_o2p","neu_o2p","con_o2p","ent_p2o","neu_p2o","con_p2o"]] = R
S["nli_bi"]  = np.minimum(S.ent_o2p, S.ent_p2o)
S["nli_con"] = np.maximum(S.con_o2p, S.con_p2o)
S[["pair_id","key","perturbation_family","severity","mean_flip","sem_preservation_score","w","ent_o2p","ent_p2o","nli_bi","nli_con"]].to_csv(f"{OUT_DIR}/nli_scores.csv", index=False)
print(S[["ent_o2p","ent_p2o","nli_bi","nli_con"]].describe().round(3))

In [ ]:
# (a) per-family fidelity profile (weighted to corpus level)  (b) flip rate restricted to NLI-equivalent pairs
def wmean(x, w): return float(np.sum(x*w)/np.sum(w))
def prof(g):
    w=g.w.values
    return pd.Series(dict(n_scored=len(g), mean_nli_bi=wmean(g.nli_bi,w), share_equiv_0p5=wmean(g.nli_bi>=.5,w), share_equiv_0p9=wmean(g.nli_bi>=.9,w),
        share_contradiction=wmean(g.nli_con>=.5,w), FR_all=wmean(g.mean_flip,w),
        FR_equiv_0p9=wmean(g[g.nli_bi>=.9].mean_flip, g[g.nli_bi>=.9].w) if (g.nli_bi>=.9).any() else np.nan,
        n_equiv_0p9=int((g.nli_bi>=.9).sum())))
P = S.groupby("perturbation_family").apply(prof).round(4)
tot = prof(S).round(4); tot.name="ALL"; P = pd.concat([P, tot.to_frame().T]); print(P.to_string()); P.to_csv(f"{OUT_DIR}/nli_family_profile.csv")
# nli_bi vs flip (descriptive, within family)
from scipy.stats import spearmanr
r = S.groupby("perturbation_family").apply(lambda g: spearmanr(g.nli_bi, g.mean_flip)[0]).round(3); print("\nSpearman(nli_bi, mean_flip) per family:\n", r.to_string())
r.to_csv(f"{OUT_DIR}/nli_flip_spearman_by_family.csv")

In [ ]:
# cluster bootstrap (over source reviews) of the corpus-level flip rate on NLI-equivalent pairs, threshold 0.9 and 0.5
rng = np.random.default_rng(1); keys, inv = np.unique(S.key.values, return_inverse=True); nk = len(keys)
def boot(mask, B=500):
    w = S.w.values*mask; y = S.mean_flip.values; est = (w*y).sum()/w.sum(); vals=[]
    for _ in range(B):
        cnt = np.bincount(rng.integers(0,nk,nk), minlength=nk)[inv]
        ww = w*cnt; vals.append((ww*y).sum()/ww.sum())
    return est, *np.percentile(vals,[2.5,97.5])
rows=[]
for name,mask in [("all scored pairs", np.ones(len(S),bool)), ("nli_bi >= 0.5", (S.nli_bi>=.5).values), ("nli_bi >= 0.9", (S.nli_bi>=.9).values)]:
    e,lo,hi = boot(mask); rows.append(dict(subset=name, n=int(mask.sum()), FR=round(e,4), ci_lo=round(lo,4), ci_hi=round(hi,4)))
B = pd.DataFrame(rows); print(B.to_string(index=False)); B.to_csv(f"{OUT_DIR}/nli_restricted_FR.csv", index=False)

In [ ]:
# (c) agreement with the human audit: NLI on the 576 audited pairs, AUC vs the human meaning label; compare with Φ
from sklearn.metrics import roc_auc_score
H = pd.read_csv(HUMAN_CSV, encoding="utf-8-sig"); HR = score(H, "human")
H["nli_bi"] = np.minimum(HR[:,0], HR[:,3])
res = []
for name,y in [("fully meaning-preserving (yes vs rest)", H.meaning_preserving=="yes"), ("not meaning-altering (not 'no')", H.meaning_preserving!="no"), ("fully natural", H.natural=="yes")]:
    if y.nunique()<2: continue
    res.append(dict(target=name, AUC_nli_bi=round(roc_auc_score(y,H.nli_bi),3), AUC_phi=round(roc_auc_score(y,H.sem_preservation_score),3)))
AUC = pd.DataFrame(res); print(AUC.to_string(index=False)); AUC.to_csv(f"{OUT_DIR}/nli_vs_human_auc.csv", index=False)
print("\nmean nli_bi by human meaning label:\n", H.groupby("meaning_preserving").nli_bi.agg(["mean","count"]).round(3))
H[["sample_id","perturbation_family","meaning_preserving","nli_bi","sem_preservation_score"]].to_csv(f"{OUT_DIR}/nli_on_human_pairs.csv", index=False)
import zipfile
with zipfile.ZipFile(f"{OUT_DIR}/nli_tables.zip","w") as z:
    for f in glob.glob(f"{OUT_DIR}/*.csv"):
        if "nli_scores" not in f: z.write(f, os.path.basename(f))
print("\nsend me:", f"{OUT_DIR}/nli_tables.zip (small)")